# 03 ・ 資料整合：把兩份清單變成一份

## 這一章要做什麼

兩家影城各有一份電影清單，中間有大量重複。這一章把它們合併成一份，
並且記錄每部片在哪幾家上映，最後用 pandas 做篩選和排序。

## 核心問題

同一部電影在兩家影城會出現兩次。要怎麼知道它們是同一部？

**用片名比對不可靠** —— 兩家寫的可能不一樣（有無副標、標點不同、
一家中文一家英文）。而且就算現在剛好一樣，明天也可能不一樣。

**用 TMDB 的電影 id 就穩了。** 上一章每部片都補上了 TMDB 資料，
裡面的 `id` 是這部電影在資料庫的唯一編號 —— 這就是跨來源的共同身分證。

## 本章產出

`movieapp/merge.py`：

```python
merge.merge_sources(enriched)   # 合併去重，記錄來源
merge.apply_filters(movies, …)  # 篩選 + 排序
merge.search_movies(movies, kw) # 關鍵字搜尋
merge.to_dataframe(movies)      # 轉成 pandas 表格
```

---
## 0 ・ 開場與備料

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")
from movieapp.config import setup; setup(requires=["sources", "tmdb"])

In [ ]:
from movieapp import sources, tmdb
from movieapp.config import pad

by_source, errors = sources.titles_by_source()
genre_names, _ = tmdb.genres()

enriched = {}
for key, titles in by_source.items():
    movies, _ = tmdb.enrich(titles, source=key, workers=8)
    enriched[key] = movies
    print(f"  {sources.SOURCES[key][0]}：{len(movies)} 部")

total = sum(len(v) for v in enriched.values())
print(f"\n兩家加起來 {total} 部（含重複）")

---
## 1 ・ 先看看重複長什麼樣

把兩家的片名擺在一起，找出 TMDB id 相同的：

In [ ]:
show_ids = {m["meta"]["id"]: m["title"] for m in enriched["showtimes"]}
mira_ids = {m["meta"]["id"]: m["title"] for m in enriched["miramar"]}
shared = set(show_ids) & set(mira_ids)

print(f"兩家都有的電影：{len(shared)} 部\n")
print(pad("秀泰的片名", 26) + pad("美麗華的片名", 26) + "TMDB id")
print("-" * 62)
for movie_id in list(shared)[:10]:
    same = "  <- 片名一樣" if show_ids[movie_id] == mira_ids[movie_id] else ""
    print(pad(show_ids[movie_id][:24], 26) + pad(mira_ids[movie_id][:24], 26)
          + str(movie_id) + same)

這一批剛好片名都一樣，但**不能靠這個** ——
只要有一家改了標點或加了副標，字串比對就會失效，
清單裡就會冒出兩筆一模一樣的電影。

用 `id` 判斷就沒有這個問題。

---
## 2 ・ 合併的規則

合併時每部片要保留三件事：

1. **TMDB 資料**（評分、海報、類型…）—— 兩家查到的是同一筆，取其一即可
2. **來源清單** —— 這部片在哪幾家上映，畫面上要標出來
3. **各家的原始片名** —— 方便對照，也方便除錯

沒有 TMDB id 的（查不到的片）就退而求其次用片名當識別，
至少不會讓它們互相蓋掉。

In [ ]:
%%writefile ../movieapp/merge.py
"""跨影城的資料整合：合併、去重、篩選、排序。

同一部電影在兩家影城會出現兩次，而且片名可能不一樣。
光比字串沒辦法判斷是不是同一部，所以要用 TMDB 的電影 id 當作共同身分證。

本檔案由 notebooks/03_資料整合.ipynb 的 %%writefile 產生。
要修改請回去改那一格，不要直接編輯這裡。
"""

import re

# 來源代號對應的顯示名稱
SOURCE_LABELS = {"showtimes": "秀泰影城", "miramar": "美麗華影城"}


def movie_key(movie):
    """一部電影的唯一身分。

    有 TMDB id 就用 id（跨影城通用），沒有的話退而求其次用片名。
    """
    tmdb_id = (movie.get("meta") or {}).get("id")
    return f"tmdb:{tmdb_id}" if tmdb_id is not None else f"title:{movie.get('title')}"


def merge_sources(enriched_by_source):
    """把各影城的電影清單合併成一份，同一部片只留一筆。

    enriched_by_source: {來源代號: [tmdb.enrich() 產生的電影, ...]}

    回傳的每一筆會多出 sources（在哪幾家上映）和 titles（各家的原始片名）。
    """
    merged = {}
    for source, movies in enriched_by_source.items():
        for movie in movies:
            key = movie_key(movie)
            existing = merged.get(key)
            if existing:
                if source not in existing["sources"]:
                    existing["sources"].append(source)
                existing["titles"].setdefault(source, movie.get("title"))
            else:
                merged[key] = {
                    "title": movie.get("title"),
                    "meta": movie.get("meta"),
                    "sources": [source],
                    "titles": {source: movie.get("title")},
                }
    return list(merged.values())


def source_labels(movie):
    """把來源代號換成中文名稱，例如 ['秀泰影城', '美麗華影城']。"""
    return [SOURCE_LABELS.get(s, s) for s in movie.get("sources", [])]


# --------------------------------------------------------------------------
# 篩選與排序
# --------------------------------------------------------------------------
def _release_month(date_str):
    """把 "2026-08-07" 換成可以直接比大小的月份序號。"""
    match = re.match(r"^(\d{4})-(\d{1,2})", str(date_str or "").strip())
    if not match:
        return None
    return int(match.group(1)) * 12 + int(match.group(2))


def apply_filters(
    movies,
    adult=None,
    genre_id=None,
    min_popularity=None,
    min_vote_average=None,
    since=None,
    sort_by=None,
    descending=True,
):
    """篩選並排序電影清單。

    adult             True/False 只留成人片或非成人片，None 不篩
    genre_id          只留包含這個類型的電影
    min_popularity    熱門度下限
    min_vote_average  評分下限
    since             上映日期下限，格式 "2026-01"
    sort_by           "popularity" / "vote_average" / "release_date"
    """
    since_month = _release_month(since) if since else None
    result = []

    for movie in movies:
        meta = movie.get("meta") or {}
        if adult is not None and bool(meta.get("adult")) is not bool(adult):
            continue
        if genre_id is not None and genre_id not in (meta.get("genre_ids") or []):
            continue
        if min_popularity is not None and (meta.get("popularity") or 0) < min_popularity:
            continue
        if min_vote_average is not None and (meta.get("vote_average") or 0) < min_vote_average:
            continue
        if since_month is not None:
            month = _release_month(meta.get("release_date"))
            if month is None or month < since_month:
                continue
        result.append(movie)

    if sort_by:
        def sort_key(movie):
            value = (movie.get("meta") or {}).get(sort_by)
            if sort_by == "release_date":
                return _release_month(value) or 0
            return value or 0

        result.sort(key=sort_key, reverse=descending)

    return result


def search_movies(movies, keyword):
    """用關鍵字比對影城片名和 TMDB 片名。"""
    query = str(keyword or "").strip().lower()
    if not query:
        return list(movies)
    return [
        m
        for m in movies
        if query in str(m.get("title", "")).lower()
        or query in str((m.get("meta") or {}).get("title", "")).lower()
    ]


def bounds(movies):
    """算出各欄位的實際範圍，畫面上的滑桿要用它決定刻度。"""
    populars = [(m.get("meta") or {}).get("popularity") or 0 for m in movies]
    years = []
    for m in movies:
        date = str((m.get("meta") or {}).get("release_date") or "")
        if len(date) >= 4 and date[:4].isdigit():
            years.append(int(date[:4]))
    return {
        "popularity_max": max(populars) if populars else 0,
        "year_min": min(years) if years else None,
        "year_max": max(years) if years else None,
        "count": len(movies),
    }


# --------------------------------------------------------------------------
# 給 pandas 用的表格形式
# --------------------------------------------------------------------------
def to_records(movies, genre_names=None):
    """把巢狀的電影資料攤平成一列一部電影，方便丟進 DataFrame。"""
    genre_names = genre_names or {}
    records = []
    for movie in movies:
        meta = movie.get("meta") or {}
        records.append(
            {
                "片名": movie.get("title"),
                "TMDB片名": meta.get("title"),
                "上映日期": meta.get("release_date") or None,
                "評分": meta.get("vote_average"),
                "評分人數": meta.get("vote_count"),
                "熱門度": meta.get("popularity"),
                "類型": "、".join(
                    genre_names.get(g, str(g)) for g in (meta.get("genre_ids") or [])
                ),
                "成人片": bool(meta.get("adult")),
                "上映影城": "、".join(source_labels(movie)),
                "tmdb_id": meta.get("id"),
            }
        )
    return records


def to_dataframe(movies, genre_names=None):
    """需要 pandas，沒安裝時給出明確訊息而不是 ImportError。"""
    try:
        import pandas as pd
    except ImportError:
        raise RuntimeError("需要 pandas，請執行 %pip install -r ../requirements.txt")
    return pd.DataFrame(to_records(movies, genre_names))

# --------------------------------------------------------------------------
# 整條流程
# --------------------------------------------------------------------------
def catalog(language="zh-TW", workers=8):
    """把整條流程串起來：抓兩家影城 -> TMDB 補資料 -> 跨來源合併去重。

    回傳 (movies, genre_names, errors)。

    網頁服務的 /api/movies/ 直接回傳這個結果，notebook 也呼叫同一個函式 ——
    兩邊跑的是同一段程式碼，不是兩份各自走鐘的複製品。
    這就是「邏輯只留一份」在這個專案裡最具體的樣子。

    任何一家影城掛掉都不會讓整份清單失敗：錯誤收集在 errors 裡回報，
    拿得到的資料照常回傳。
    """
    from movieapp import sources, tmdb

    by_source, errors = sources.titles_by_source()

    genre_names, genre_error = tmdb.genres(language=language)
    if genre_error:
        errors["TMDB 類型"] = genre_error

    enriched = {}
    for key, titles in by_source.items():
        movies, failures = tmdb.enrich(titles, language=language, workers=workers, source=key)
        enriched[key] = movies
        if failures:
            label = SOURCE_LABELS.get(key, key)
            errors[f"{label} TMDB 查詢"] = next(iter(failures.values()))

    return merge_sources(enriched), genre_names, errors

In [ ]:
from movieapp import merge

movies = merge.merge_sources(enriched)
both = [m for m in movies if len(m["sources"]) > 1]

print(f"合併前 {total} 部 -> 去重後 {len(movies)} 部")
print(f"其中 {len(both)} 部兩家都有上映\n")

for m in both[:8]:
    print(f"  {pad(m['meta']['title'][:22], 26)} {'、'.join(merge.source_labels(m))}")

---
## 3 ・ 轉成表格

到目前為止電影是巢狀的 dict，看起來很吃力。
攤平成一列一部電影之後，就能用 pandas 處理了。

In [ ]:
df = merge.to_dataframe(movies, genre_names)
print(f"{len(df)} 列 x {len(df.columns)} 欄")
df.head(10)

In [ ]:
# 有了 DataFrame，一行就能看出資料的樣貌
df[["評分", "評分人數", "熱門度"]].describe().round(2)

---
## 4 ・ 篩選與排序

`apply_filters()` 把畫面上那些篩選條件寫成函式。
每個條件都是獨立的，可以任意組合。

In [ ]:
# 評分 7 分以上，依熱門度排序
top = merge.apply_filters(movies, min_vote_average=7.0, sort_by="popularity")
print(f"評分 7 分以上：{len(top)} 部\n")
for m in top[:8]:
    meta = m["meta"]
    print(f"  {pad(meta['title'][:24], 26)} 評分 {meta['vote_average']:.1f}"
          f"  熱門度 {meta.get('popularity', 0):>7.0f}")

In [ ]:
# 找出動畫類型的電影
animation_id = next((k for k, v in genre_names.items() if v == "動畫"), None)
animations = merge.apply_filters(movies, genre_id=animation_id, sort_by="vote_average")
print(f"動畫（類型 id {animation_id}）：{len(animations)} 部")
for m in animations[:6]:
    print(f"  {m['meta']['title']}（{m['meta'].get('release_date') or '未定'}）")

In [ ]:
# 同樣的事情用 pandas 做 —— 語法完全不同，結果一樣
df[df["類型"].str.contains("動畫", na=False)].sort_values("評分", ascending=False).head(6)

兩種寫法各有好處：

- `apply_filters()` 是**純 Python**，網頁服務直接用得上，沒有額外相依
- **pandas** 適合在 notebook 裡探索資料、做統計、畫圖

`merge.py` 兩種都提供，因為它們的使用場合不一樣。

---
## 5 ・ 關鍵字搜尋與範圍

搜尋要同時比對影城片名和 TMDB 片名，因為兩者可能不一樣。

In [ ]:
for keyword in ["蜘蛛", "驀然", "spider"]:
    found = merge.search_movies(movies, keyword)
    print(f"  搜尋 {keyword!r}：{len(found)} 部 -> "
          f"{[m['meta']['title'] for m in found[:3]]}")

print()
print("資料範圍（畫面上的滑桿刻度就是用這個算的）：")
for key, value in merge.bounds(movies).items():
    print(f"  {key:16} {value}")

---
## 6 ・ 把整條流程收成一個函式

從第 0 節到現在，「拿到一份乾淨的電影清單」要走這些步驟：

```
sources.titles_by_source()   兩家影城的片名
        ↓
tmdb.enrich()                每一部補上 TMDB 資料
        ↓
merge.merge_sources()        跨來源合併去重
```

這三步每次都一樣。第 5 章的網頁服務也要走同一條路 ——
**如果讓它自己再寫一次，就會有兩份各自走鐘的流程。**

所以收成 `merge.catalog()`：notebook 呼叫它，網頁服務也呼叫它，
兩邊跑的是同一段程式碼。這就是「邏輯只留一份」在這個專案裡最具體的樣子。

In [ ]:
movies_again, genres_again, errors_again = merge.catalog()

print(f"catalog() 一次拿到 {len(movies_again)} 部電影、{len(genres_again)} 種類型")
print("錯誤：", errors_again or "無")
print()
print("跟前面手動走三步的結果一致？",
      {m["meta"]["id"] for m in movies_again} == {m["meta"]["id"] for m in movies})

---
## 小結

- 跨來源比對要用**穩定的識別碼**（TMDB id），不要用會變動的字串
- 合併時保留來源資訊，畫面上才標得出「哪幾家有上映」
- 同一套篩選邏輯提供兩種形式：純 Python 給服務用，pandas 給分析用
- 重複出現的流程收成一個函式（`catalog()`），notebook 和服務才不會各寫一份

### 產出

`movieapp/merge.py`

### 下一章

**04_Gemini對話** —— 現在有一份乾淨的電影清單了。
下一章把它交給 AI，讓它根據**今天實際上映的電影**回答問題，
而不是憑記憶亂講。